## Silver: fct_eeg_zahlung_jaehrlich.

Annual EEG payment per Anlage, joined to dim_anlage for geography.

Source: bronze.ntp_jahresabrechnung_bewegungsdaten (8.18M payment rows for 2024).
Output: silver.fct_eeg_zahlung_jaehrlich (~1-2M rows, one per Anlage receiving payments).

Transformation rules:
  - Filter out veraeusserungsform=5 (Sonstiges/bookkeeping; eeg_zahlung is always 0)
  - Cast German decimals: '1054,35' -> 1054.35
  - Sum eeg_zahlung per (eeg_mastr_nr, jahr)
  - Left-join to silver.dim_anlage on eeg_mastr_nr = eeg_mastr_nummer
  - Quarantine unmatched rows in fct_eeg_zahlung_jaehrlich_unmatched

Quality gate: >95% of EEG € must match an Anlage in dim_anlage.

In [0]:
%sql
DROP TABLE eeg_dev.silver.fct_eeg_zahlung_jaehrlich;
DROP TABLE eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched;

In [0]:
%sql
-- Test: is the Sauerlach geothermal plant actually joinable?
WITH sauerlach AS (
  SELECT 'EEG989741749877' AS payment_id
)
SELECT
  p.payment_id,
  d_eeg.bundesland AS via_eeg,
  d_mastr.bundesland AS via_mastr
FROM sauerlach p
LEFT JOIN eeg_dev.silver.dim_anlage d_eeg
  ON p.payment_id = d_eeg.eeg_mastr_nummer
LEFT JOIN eeg_dev.silver.dim_anlage d_mastr
  ON p.payment_id = d_mastr.mastr_nummer;

  SELECT payment_key, verguetung_eur, technologie, bundesland
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich
WHERE payment_key = 'EEG989741749877';

SELECT payment_key, ROUND(verguetung_eur / 1e6, 2) AS millionen_eur,
       ROUND(strommenge_kwh / 1e6, 1) AS gwh
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
WHERE uenb = 'tennet'
ORDER BY verguetung_eur DESC
LIMIT 20;

SELECT payment_key, ROUND(verguetung_eur / 1e6, 2) AS millionen_eur
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
WHERE uenb = '50hertz'
ORDER BY verguetung_eur DESC
LIMIT 5;

In [0]:
from pyspark.sql import functions as F


# ---------- 1. Read bronze + filter + cast decimals ---------------------------

bronze = spark.table("eeg_dev.bronze.ntp_jahresabrechnung_bewegungsdaten")

clean = (
    bronze
    .filter(F.col("veraeusserungsform") != "5")
    .filter(F.col("eeg_mastr_nr").isNotNull())
    .withColumn(
        "eeg_zahlung_eur",
        F.regexp_replace(F.col("eeg_zahlung"), ",", ".").cast("double"),
    )
    .withColumn(
        "eeg_einnahmen_eur",
        F.regexp_replace(F.col("eeg_einnahmen"), ",", ".").cast("double"),
    )
    .withColumn(
        "strommenge_kwh",
        F.regexp_replace(F.col("strommenge"), ",", ".").cast("double"),
    )
)


# ---------- 2. Aggregate per (Anlage, year) -----------------------------------

aggregated = (
    clean
    .groupBy("eeg_mastr_nr", "_jahr", "_uenb")
    .agg(
        F.sum("eeg_zahlung_eur").alias("verguetung_eur"),
        F.sum("eeg_einnahmen_eur").alias("einnahmen_eur"),
        F.sum("strommenge_kwh").alias("strommenge_kwh"),
        F.count("*").alias("zeilen_anzahl"),
    )
    .withColumnRenamed("_jahr", "jahr")
    .withColumnRenamed("_uenb", "uenb")
)


# ---------- 3. Join to dim_anlage with dual-key fallback ----------------------

# dim_anlage has BOTH keys: mastr_nummer (SEE...) and eeg_mastr_nummer (EEG...)
dim = spark.table("eeg_dev.silver.dim_anlage").select(
    F.col("mastr_nummer"),
    F.col("eeg_mastr_nummer"),
    F.col("technologie"),
    F.col("bundesland"),
    F.col("landkreis"),
    F.col("gemeinde"),
    F.col("gemeindeschluessel"),
    F.col("plz"),
    F.col("brutto_leistung_kw"),
    F.col("inbetriebnahme_datum"),
    F.col("betriebs_status"),
)

# Join logic: payment ID matches EITHER key in dim_anlage.
# This handles:
#   - EEG... prefix → matches eeg_mastr_nummer (most rows, esp. wind/solar/biomass)
#   - SEE... prefix → matches mastr_nummer (gsgk geothermal, deleted units)
#   - KWK... prefix → won't match either; correctly stays unmatched (KWKG, not EEG)
joined = (
    aggregated.alias("a")
    .join(
        dim.alias("d"),
        (F.col("a.eeg_mastr_nr") == F.col("d.eeg_mastr_nummer"))
        | (F.col("a.eeg_mastr_nr") == F.col("d.mastr_nummer")),
        how="left",
    )
    .select(
        F.col("a.eeg_mastr_nr").alias("payment_key"),
        F.col("d.mastr_nummer"),
        F.col("d.eeg_mastr_nummer"),
        F.col("a.jahr"),
        F.col("a.uenb"),
        F.col("d.technologie"),
        F.col("d.bundesland"),
        F.col("d.landkreis"),
        F.col("d.gemeinde"),
        F.col("d.gemeindeschluessel"),
        F.col("d.plz"),
        F.col("d.brutto_leistung_kw"),
        F.col("d.inbetriebnahme_datum"),
        F.col("d.betriebs_status"),
        F.col("a.verguetung_eur"),
        F.col("a.einnahmen_eur"),
        F.col("a.strommenge_kwh"),
        F.col("a.zeilen_anzahl"),
        F.when(F.col("d.bundesland").isNotNull(), F.lit(True))
            .otherwise(F.lit(False))
            .alias("ist_zugeordnet"),
    )
)


# ---------- 4. Split: matched vs unmatched -----------------------------------

matched = joined.filter(F.col("ist_zugeordnet") == True).drop("ist_zugeordnet")
unmatched = joined.filter(F.col("ist_zugeordnet") == False).drop("ist_zugeordnet")


# ---------- 5. Write tables ---------------------------------------------------

(
    matched.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("eeg_dev.silver.fct_eeg_zahlung_jaehrlich")
)

(
    unmatched.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched")
)


# ---------- 6. The acceptance test --------------------------------------------

stats = (
    joined
    .groupBy("ist_zugeordnet")
    .agg(
        F.count("*").alias("anlagen"),
        F.sum("verguetung_eur").alias("verguetung_eur"),
    )
    .collect()
)

total_anlagen = sum(r["anlagen"] for r in stats)
total_eur = sum(r["verguetung_eur"] or 0 for r in stats)
matched_anlagen = next((r["anlagen"] for r in stats if r["ist_zugeordnet"]), 0)
matched_eur = next((r["verguetung_eur"] for r in stats if r["ist_zugeordnet"]), 0) or 0

print("=" * 60)
print("Match rate report")
print("=" * 60)
print(f"Total unique Anlagen with payments:  {total_anlagen:>12,}")
print(f"Matched to dim_anlage:               {matched_anlagen:>12,}  "
      f"({100 * matched_anlagen / total_anlagen:.2f}%)")
print(f"Unmatched (quarantined):             {total_anlagen - matched_anlagen:>12,}  "
      f"({100 * (total_anlagen - matched_anlagen) / total_anlagen:.2f}%)")
print()
print(f"Total Vergütung € (all):             {total_eur:>15,.0f}")
print(f"Matched €:                           {matched_eur:>15,.0f}  "
      f"({100 * matched_eur / total_eur:.2f}%)")
print(f"Unmatched €:                         {total_eur - matched_eur:>15,.0f}  "
      f"({100 * (total_eur - matched_eur) / total_eur:.2f}%)")
print()
print("Acceptance test: matched € share must be > 95%")
print(f"Result: {'PASS' if matched_eur / total_eur > 0.95 else 'FAIL'}")

In [0]:
%sql
SELECT mastr_nummer, eeg_mastr_nummer, bundesland, brutto_leistung_kw
FROM eeg_dev.silver.dim_anlage
WHERE technologie = 'gsgk'
ORDER BY brutto_leistung_kw DESC NULLS LAST
LIMIT 10;

SELECT mastr_nummer, eeg_mastr_nummer, technologie, bundesland, gemeinde
FROM eeg_dev.silver.dim_anlage
WHERE mastr_nummer = 'SEE966733125161'
   OR eeg_mastr_nummer = 'EEG989741749877';

   SELECT payment_key, verguetung_eur
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
WHERE payment_key = 'EEG989741749877'
   OR payment_key = 'SEE966733125161';

In [0]:
%sql
-- Sanity check: row counts
SELECT
  (SELECT COUNT(*) FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich) AS matched,
  (SELECT COUNT(*) FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched) AS unmatched;

-- The headline: total Vergütung 2024 by Bundesland
SELECT
  bundesland,
  ROUND(SUM(verguetung_eur) / 1e9, 2) AS milliarden_eur,
  COUNT(*) AS anlagen
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich
GROUP BY bundesland
ORDER BY milliarden_eur DESC;

-- Sanity check by technology
SELECT
  technologie,
  ROUND(SUM(verguetung_eur) / 1e9, 2) AS milliarden_eur,
  COUNT(*) AS anlagen
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich
GROUP BY technologie
ORDER BY milliarden_eur DESC;

-- Per-ÜNB breakdown of unmatched
SELECT
  uenb,
  COUNT(*) AS anlagen,
  ROUND(SUM(verguetung_eur) / 1e6, 1) AS millionen_eur,
  ROUND(AVG(verguetung_eur), 0) AS avg_eur
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
GROUP BY uenb
ORDER BY millionen_eur DESC;

SELECT
  eeg_mastr_nummer,
  uenb,
  ROUND(verguetung_eur, 0) AS eur,
  ROUND(strommenge_kwh / 1000, 0) AS mwh,
  zeilen_anzahl
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
ORDER BY verguetung_eur DESC
LIMIT 20;

-- Replace 'EEG...' with one of the actual top unmatched IDs
SELECT 'wind' AS tbl, COUNT(*) FROM eeg_dev.bronze.mastr_wind WHERE EegMastrNummer = 'EEG989741749877'
UNION ALL SELECT 'solar', COUNT(*) FROM eeg_dev.bronze.mastr_solar WHERE EegMastrNummer = 'EEG989741749877'
UNION ALL SELECT 'biomass', COUNT(*) FROM eeg_dev.bronze.mastr_biomass WHERE EegMastrNummer = 'EEG989741749877'
UNION ALL SELECT 'hydro', COUNT(*) FROM eeg_dev.bronze.mastr_hydro WHERE EegMastrNummer = 'EEG989741749877';


-- 10 unmatched Amprion IDs
SELECT 'unmatched' AS source, eeg_mastr_nummer, verguetung_eur
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
WHERE uenb = 'amprion'
ORDER BY verguetung_eur DESC
LIMIT 10;

-- 10 matched Amprion IDs for comparison
SELECT 'matched' AS source, eeg_mastr_nummer, verguetung_eur
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich
WHERE uenb = 'amprion'
ORDER BY verguetung_eur DESC
LIMIT 10;

SELECT 'solar_einheit'   AS where_found, COUNT(*) AS n FROM eeg_dev.bronze.mastr_solar   WHERE EinheitMastrNummer = 'SEE982537985732'
UNION ALL SELECT 'solar_eeg',       COUNT(*) FROM eeg_dev.bronze.mastr_solar   WHERE EegMastrNummer     = 'SEE982537985732'
UNION ALL SELECT 'wind_einheit',    COUNT(*) FROM eeg_dev.bronze.mastr_wind    WHERE EinheitMastrNummer = 'SEE982537985732'
UNION ALL SELECT 'wind_eeg',        COUNT(*) FROM eeg_dev.bronze.mastr_wind    WHERE EegMastrNummer     = 'SEE982537985732'
UNION ALL SELECT 'biomass_einheit', COUNT(*) FROM eeg_dev.bronze.mastr_biomass WHERE EinheitMastrNummer = 'SEE982537985732'
UNION ALL SELECT 'biomass_eeg',     COUNT(*) FROM eeg_dev.bronze.mastr_biomass WHERE EegMastrNummer     = 'SEE982537985732'
UNION ALL SELECT 'hydro_einheit',   COUNT(*) FROM eeg_dev.bronze.mastr_hydro   WHERE EinheitMastrNummer = 'SEE982537985732'
UNION ALL SELECT 'hydro_eeg',       COUNT(*) FROM eeg_dev.bronze.mastr_hydro   WHERE EegMastrNummer     = 'SEE982537985732';

SELECT 'solar' AS tbl, EinheitMastrNummer, EegMastrNummer
FROM eeg_dev.bronze.mastr_solar
WHERE EinheitMastrNummer LIKE '%982537985732%'
   OR EegMastrNummer LIKE '%982537985732%'
LIMIT 5;

SELECT
  SUBSTR(EinheitMastrNummer, 1, 3) AS prefix,
  COUNT(*) AS n
FROM eeg_dev.bronze.mastr_solar
GROUP BY prefix
ORDER BY n DESC;

SELECT
  uenb,
  SUBSTR(eeg_mastr_nummer, 1, 3) AS prefix,
  COUNT(*) AS n,
  ROUND(SUM(verguetung_eur) / 1e6, 1) AS millionen_eur
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
GROUP BY uenb, prefix
ORDER BY uenb, n DESC;


-- 1. What prefixes appear in mastr_solar's EinheitMastrNummer?
SELECT
  SUBSTR(EinheitMastrNummer, 1, 3) AS prefix,
  COUNT(*) AS n
FROM eeg_dev.bronze.mastr_solar
WHERE EinheitMastrNummer IS NOT NULL
GROUP BY prefix
ORDER BY n DESC
LIMIT 10;

-- 2. What prefixes appear in unmatched payments, broken by ÜNB?
SELECT
  uenb,
  SUBSTR(eeg_mastr_nummer , 1, 3) AS prefix,
  COUNT(*) AS anlagen,
  ROUND(SUM(verguetung_eur) / 1e6, 1) AS millionen_eur
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
GROUP BY uenb, prefix
ORDER BY uenb, anlagen DESC;

SELECT eeg_mastr_nummer, ROUND(verguetung_eur / 1e6, 2) AS millionen_eur
FROM eeg_dev.silver.fct_eeg_zahlung_jaehrlich_unmatched
WHERE uenb = 'tennet' AND eeg_mastr_nummer LIKE 'EEG%'
ORDER BY verguetung_eur DESC
LIMIT 5;